In [31]:
import pandas as pd
import numpy as np

#전처리(인코딩)
from sklearn.preprocessing import LabelEncoder
#전처리(스케일링)
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import RobustScaler

#전처리(검증 데이터나누기)
from sklearn.model_selection import train_test_split

#모델 - 랜덤포레스트
from sklearn.ensemble import RandomForestClassifier

#평가 -f1스코어
from sklearn.metrics import f1_score

In [32]:
train = pd.read_csv('drug_train.csv')
test = pd.read_csv('drug_test.csv')

print('train :', train.shape)
print('test :', test.shape)

train : (100, 6)
test : (100, 5)


In [33]:
set(train) - set(test)

{'Drug'}

In [34]:
target = train.pop('Drug')
target.value_counts()

Drug
DrugY    41
drugX    34
drugA    13
drugB     8
drugC     4
Name: count, dtype: int64

In [35]:
train.isnull().sum()

Age            0
Sex            0
BP             0
Cholesterol    0
Na_to_K        0
dtype: int64

In [36]:
test.isnull().sum()

Age            0
Sex            0
BP             0
Cholesterol    0
Na_to_K        0
dtype: int64

In [37]:
train.head()

,Age,Sex,BP,Cholesterol,Na_to_K
0,70,M,HIGH,HIGH,9.849
1,36,M,LOW,NORMAL,11.424
2,23,F,HIGH,HIGH,25.355
3,40,F,NORMAL,HIGH,10.103
4,45,M,LOW,NORMAL,10.017


In [38]:
obj_cols = train.select_dtypes(include='O').columns

#카테고리 확인
print(train[obj_cols].nunique())
print(test[obj_cols].nunique())

Sex            2
BP             3
Cholesterol    2
dtype: int64
Sex            2
BP             3
Cholesterol    2
dtype: int64


In [39]:
#obj 컬럼 카테고리 동일여부 확인
for col in obj_cols:
    is_same = False

    if set(train[col]) == set(test[col]):
        is_same = True
        print('col :', is_same)

col : True
col : True
col : True


In [40]:
train = pd.get_dummies(train)
test = pd.get_dummies(test)

### 데이터 나누기

In [41]:
x_train,x_val,y_train,y_val = train_test_split(
    train,
    target,
    test_size=0.2,
    random_state=0,
)

print('x_train :', x_train.shape)
print('x_val :', x_val.shape)
print('y_train :', y_train.shape)
print('y_val :', y_val.shape)

x_train : (80, 9)
x_val : (20, 9)
y_train : (80,)
y_val : (20,)


### 모델학습

In [42]:
model = RandomForestClassifier(random_state=0)
model.fit(x_train,y_train)

pred = model.predict(x_val)
pred

array(['DrugY', 'drugX', 'DrugY', 'DrugY', 'drugX', 'DrugY', 'drugX',
       'DrugY', 'drugX', 'drugX', 'drugX', 'drugA', 'DrugY', 'DrugY',
       'drugC', 'drugA', 'drugB', 'drugX', 'DrugY', 'DrugY'], dtype=object)

In [43]:
f1 = f1_score(y_val,pred, average='micro')
print("f1_score :", f1)

f1_score : 1.0


In [44]:
submit = pd.DataFrame({'pred' : pred})
submit.to_csv('drug_result.csv', index=False)
submit.head()

,pred
0,DrugY
1,drugX
2,DrugY
3,DrugY
4,drugX


### 크로스 벨리데이션

In [45]:
from sklearn.model_selection import cross_val_score

In [48]:
f1_scores = cross_val_score(model,train,target,cv=3, scoring='f1_macro')
print(f1_scores)

[1.         0.93777778 0.78461538]


In [49]:
print('cross-validation :', f1_scores.mean())

cross-validation : 0.9074643874643874
